# ASHRAE GEPIII – Modeling

Per-meter LightGBM on `log1p(meter_reading)` with a time-based holdout.


In [ ]:
import sys, pathlib, gc
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import lightgbm as lgb

from src.features import build, FEATURES, CAT_FEATURES, rmsle

In [2]:
train = build("train")
train["target"] = np.log1p(train["meter_reading"].astype("float32"))

NameError: name 'build' is not defined

## Time-based split — last 30 days as validation


In [ ]:
cutoff = train["timestamp"].max() - pd.Timedelta(days=30)
tr_idx = train["timestamp"] <= cutoff
val_idx = ~tr_idx

## Train one LightGBM per meter type


In [ ]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 255,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 5,
    "min_data_in_leaf": 200,
    "verbose": -1,
}

models = {}
val_scores = {}

for meter in sorted(train["meter"].unique()):
    mask = train["meter"] == meter
    tr = train[mask & tr_idx]
    va = train[mask & val_idx]

    dtrain = lgb.Dataset(tr[FEATURES], tr["target"], categorical_feature=CAT_FEATURES)
    dval = lgb.Dataset(va[FEATURES], va["target"], categorical_feature=CAT_FEATURES, reference=dtrain)

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        valid_sets=[dtrain, dval],
        valid_names=["train", "val"],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
    )
    pred = np.expm1(model.predict(va[FEATURES], num_iteration=model.best_iteration))
    val_scores[meter] = rmsle(va["meter_reading"].values, pred)
    models[meter] = model
    print(f"meter={meter} RMSLE={val_scores[meter]:.4f}")

print("overall val RMSLE:", np.mean(list(val_scores.values())))

## Feature importance (electricity)


In [ ]:
imp = pd.DataFrame({
    "feature": FEATURES,
    "gain": models[0].feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)
imp.head(20)

## Predict on test and write submission


In [ ]:
del train; gc.collect()

test = build("test")
preds = np.zeros(len(test), dtype="float32")
for meter, model in models.items():
    mask = (test["meter"] == meter).values
    if mask.any():
        preds[mask] = np.expm1(
            model.predict(test.loc[mask, FEATURES], num_iteration=model.best_iteration)
        )

preds = np.clip(preds, 0, None)
submission = pd.DataFrame({"row_id": test["row_id"], "meter_reading": np.round(preds, 4)})
submission.to_csv("../submission.csv", index=False)
submission.head()
